# OE5: comprobación reproducible en Google Colab

**Objetivo oficial (literal).** Determinar el nivel de precisión de la adaptación multimodal del artefacto desarrollado, calculado automáticamente como F1_adapt frente a la referencia gold, contrastándolo con el valor de referencia establecido de 0,85.

**Qué comprueba este cuaderno.** Recalcula el F1 por caso desde el vector de selección de cada ciclo y la referencia gold sellada, el F1_adapt por perfil, los intervalos, las pruebas unilaterales, la Enmienda 1 y la sensibilidad a la regla de inclusión, y los compara con el resultado local. Reproducir el cálculo no es volver a generar los 1 000 ciclos.

**Datos de entrada.** Ciclos de la campaña (10 réplicas por perfil con el vector de selección), plan, perfiles, referencia gold sellada y los dos scripts de análisis, con los resultados locales para comparar.

**Qué no hace.** No regenera los 1 000 ciclos ni verifica la independencia del F1 respecto del equipo. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Ejecute las celdas en orden en un entorno nuevo de Colab; la carga de datos pide subir los archivos indicados.

**Contenido (diez pasos).** 1 descripción; 2 datos de entrada; 3 verificación de archivos y estructura; 4 dependencias; 5 ejecución del análisis; 6 reconstrucción de los indicadores; 7 comparación con los resultados oficiales; 8 diferencias numéricas; 9 verificación de las salidas finales; 10 conclusión de reproducibilidad (calculada por la última celda).

In [ ]:
import sys, platform
print('Python', sys.version)
print(platform.platform())
!pip -q install numpy scipy
import numpy, scipy; print('numpy', numpy.__version__, 'scipy', scipy.__version__)
!nproc; !free -g | head -2

In [ ]:
import os, zipfile
if not os.path.exists('oe5_colab_paquete.zip'):
    from google.colab import files
    files.upload()
zipfile.ZipFile('oe5_colab_paquete.zip').extractall('oe5')
print(sorted(os.listdir('oe5')), len(os.listdir('oe5/campana_01')), 'archivos de la campaña')

In [ ]:
import hashlib
base='oe5/campana_01'; bad=[]
for l in open(base+'/SHA256SUMS.txt'):
    h,n=l.split(None,1); n=n.strip().lstrip('*')
    if hashlib.sha256(open(os.path.join(base,n),'rb').read()).hexdigest()!=h: bad.append(n)
print('archivos con hash distinto:', bad); assert not bad

In [ ]:
!python oe5/analisis_oe5.py oe5/campana_01 oe5/salida_colab > oe5/log_analisis.txt 2>&1
!python oe5/analisis_oe5_enmienda1.py oe5/campana_01 oe5/salida_colab_enmienda1 > oe5/log_enmienda1.txt 2>&1
!tail -3 oe5/log_analisis.txt oe5/log_enmienda1.txt

In [ ]:
import json
a=json.load(open('oe5/salida_colab/resultados_oe5.json')); b=json.load(open('oe5/esperado_resultados_oe5_local.json'))
for k in ['media','de','dz','shapiro_p','t_p_mayor','t_p_menor','wilcoxon_p_mayor']:
    print(k, a['principal'][k], b['principal'][k], 'dif=%.2e'%abs(a['principal'][k]-b['principal'][k]))
print('IC95 t', a['principal']['ic95_t'], b['principal']['ic95_t'])
print('bootstrap IC (semilla fija)', a['principal']['ic95_bootstrap'], b['principal']['ic95_bootstrap'])
print('sensibilidad', a['sensibilidad_regla_inclusion'])
c=json.load(open('oe5/salida_colab_enmienda1/resultados_enmienda1.json')); d=json.load(open('oe5/esperado_enmienda1_local.json'))
for k in ['p_bootstrap_t_mayor','p_t_mayor']: print(k, c['prueba_de_la_media'][k], d['prueba_de_la_media'][k])
print('ICC', c['dependencia_replicas']['icc'], d['dependencia_replicas']['icc'])
print('HE5 respaldada:', c['decision']['HE5_mu_mayor_0.85_respaldada'], '| RNF03 cumplido:', c['decision']['RNF03_cumplido'])
print('valor principal conservado:', c['decision']['valor_principal_conservado'])

## Conclusión de reproducibilidad

In [ ]:
import json, os
def _hojas(x, p=''):
    if isinstance(x, dict):
        for k, v in x.items(): yield from _hojas(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i, v in enumerate(x): yield from _hojas(v, p+'/'+str(i))
    else: yield p, x
def _verif(a, b):
    A = dict(_hojas(json.load(open(a, encoding='utf8')))); B = dict(_hojas(json.load(open(b, encoding='utf8'))))
    mx = 0.0; ng = 0; malas = []
    for k in A:
        if k in B and isinstance(A[k], (int, float)) and isinstance(B[k], (int, float)) and not isinstance(A[k], bool) and not isinstance(B[k], bool):
            d = abs(A[k]-B[k]); mx = max(mx, d)
            if d > 1e-9 and d/max(abs(B[k]), 1e-300) > 1e-9: ng += 1; malas.append((k, A[k], B[k]))
        elif k in B and A[k] != B[k]: ng += 1; malas.append((k, A[k], B[k]))
    return set(A) == set(B), len(A), mx, ng, malas
CHECKS = [('OE5 resultados_oe5.json', 'oe5/salida_colab/resultados_oe5.json', 'oe5/esperado_resultados_oe5_local.json'), ('OE5 enmienda 1', 'oe5/salida_colab_enmienda1/resultados_enmienda1.json', 'oe5/esperado_enmienda1_local.json')]
HTTP = []
print('CONCLUSION DE REPRODUCIBILIDAD')
todo_ok = True
dep_version = False
for etiqueta, a, b in CHECKS:
    mismas, n, mx, ng, malas = _verif(a, b)
    solo_wilcoxon = ng > 0 and all(k.endswith('/p_wilcoxon') for k, _, _ in malas)
    ok = mismas and ng == 0; todo_ok &= ok or (mismas and solo_wilcoxon); dep_version |= (not ok) and mismas and solo_wilcoxon
    veredicto = 'COINCIDE' if ok else ('COINCIDE salvo p_wilcoxon (aproximación normal que depende de la versión de scipy)' if mismas and solo_wilcoxon else 'NO COINCIDE')
    print(f'  {etiqueta}: misma estructura {mismas} | hojas {n} | diferencia absoluta máxima {mx:.3e} | hojas con diferencia > 1e-9: {ng} -> {veredicto}')
    for k, x, y in malas[:10]: print(f'     hoja distinta: {k} | Colab {x!r} | local {y!r}')
for k in HTTP:
    r = json.load(open(f'http/out/resumen_{k}.json'))
    ok = r['corridas_validas_finales_recalculadas'] == r['corridas_validas_finales_con_todos_los_campos_iguales'] and max(r['diferencia_abs_max_por_campo'].values()) == 0
    todo_ok &= ok
    print(f"  {k}: indicadores por corrida recalculados desde las peticiones: {r['corridas_validas_finales_recalculadas']} corridas válidas, {r['corridas_validas_finales_con_todos_los_campos_iguales']} idénticas -> {'COINCIDE' if ok else 'NO COINCIDE'}")
if todo_ok and dep_version: print('Resultado: el análisis se reproduce con los datos oficiales; todas las hojas coinciden dentro de 1e-9 salvo los p de Wilcoxon listados arriba, que dependen de la versión de scipy instalada (scipy', __import__('scipy').__version__, 'en este entorno). Las demás cantidades, incluido el ajuste de Holm calculado con p_t, coinciden.')
else: print('Resultado:', 'el análisis se reproduce con los datos oficiales y coincide con el resultado local dentro de la tolerancia de 1e-9 (las diferencias de coma flotante se informan arriba).' if todo_ok else 'HAY DIFERENCIAS: revise las líneas marcadas como NO COINCIDE antes de concluir.')
print('Alcance: se vuelve a ejecutar el análisis estadístico sobre los datos de entrada; no se repiten las campañas ni se regeneran los ciclos o las corridas, y la reproducción no implica que las condiciones de adquisición fueran controladas.')